# Wind Energy

In [ ]:
import scipy.stats as sps
from scipy.special import gamma
import numpy as np
import matplotlib.pyplot as plt
import enn554.wind as wind
from enn554.paths import data_dir, outputs_dir
dd = data_dir()
od = outputs_dir()

## Method of bins

In [ ]:
coopers_gap = wind.merra_wind_speed_data()
coopers_gap.import_data(dd/"tutorial_4/POWER_Point_Hourly_20150101_20241231_026d73S_151d47E_UTC.csv")
coopers_gap.utc_to_lst(10) # we only really need to do this if we are aligning with the load.

fig,ax = plt.subplots()
counts, bin_edges, patches = ax.hist(coopers_gap.data['WS50M'])
m = 0.5 * (bin_edges[1:] + bin_edges[:-1])
ax.scatter(m,np.zeros_like(m),marker='*',color='red')

In [ ]:
u_cutin  = 3.5
u_rated  = 13.5
u_cutout = 25
p_rated  = 1000.0  # kW

def power_curve(u):
    p = np.where(
        (u >= u_cutin) & (u < u_rated),
        p_rated * ((u - u_cutin) / (u_rated - u_cutin)) ** 3,
        np.where(
            (u >= u_rated) & (u <= u_cutout),
            p_rated,
            0.0
        )
    )
    return p

In [ ]:
sorted_ws = coopers_gap.data['WS50M'].values
sorted_ws = np.flip(np.sort(sorted_ws))
hours_above = np.arange(1, len(sorted_ws)+1)
fig, ax = plt.subplots()
ax.plot(8760*hours_above/len(sorted_ws),sorted_ws) # scale x-axis to hours per year
ax.set_xlabel("Hours per year")
ax.set_ylabel("Wind Speed (m/s)")
ax.set_title("Speed duration curve at 100m")


In [ ]:
ws = coopers_gap.data['WS50M'].values
turbine_power = power_curve(sorted_ws)
hours_above = np.arange(1, len(turbine_power)+1)
fig, ax = plt.subplots()
ax.plot(8760*hours_above/len(turbine_power),turbine_power) # scale x-axis to hours per year
ax.set_xlabel("Hours per year")
ax.set_ylabel("Wind Power (kW)")
ax.set_title("Power duration curve")

## Testing the average power formula

In [ ]:
c,k = 100,2.0
ρ=1.22
N = 1000000
u = sps.weibull_min(k,scale=c).rvs(N)
WPD = 0.5*ρ*(u**3).mean()
WPD_analytical = 0.5*ρ*c**3*gamma(1+3.0/k)
print(f'Monte Carlo: {WPD:.2f}, Analytical: {WPD_analytical:.2f}')
print(f'Representative: {0.5*ρ*u.mean()**3:.2f}')

In [ ]:
# Power curve parameters
plt.rcParams.update({
    'font.size':        14,   # default for all text
    'axes.titlesize':   16,   # axes title
    'axes.labelsize':   14,   # x and y labels
    'xtick.labelsize':  12,   # tick labels
    'ytick.labelsize':  12,
    'legend.fontsize':  12,
    'figure.titlesize': 18,   # suptitle
})


u = np.linspace(0, 27, 500)
p = power_curve(u)

fig, ax = plt.subplots(figsize=(8, 5))
ax2 = ax.twinx()
ax.plot(u, p, color='gray', linewidth=2,label='Power curve')

ax.set_xlabel('Wind speed (m/s)')
ax.set_ylabel(r'$P_w$')
ax.set_xlim(0, 27)

ax.set_ylim(-100, 1500)
ax.set_xticks([5, 10, 15, 20, 25])
ax.set_yticks([0, 250, 500, 750, 1000, 1250])
ax.set_facecolor('#e8e8e8')
ax.grid(True, color='white', linewidth=0.8)

ax.annotate('Cut-in',  xy=(u_cutin,  0),    xytext=(u_cutin,  40),  fontsize=12)
ax.annotate('Rated',   xy=(u_rated,  p_rated), xytext=(u_rated-1, p_rated+35), fontsize=12)
ax.annotate('Cut-out', xy=(u_cutout, p_rated), xytext=(u_cutout-1, p_rated+50),
            fontsize=12, va='center')

counts, bin_edges, patches = ax2.hist(coopers_gap.data['WS50M']+3.0,bins=100,density=True,alpha=0.25,label='data')
ax2.set_yticks([])
fig.legend(loc='upper right',bbox_to_anchor=(0.95,0.95))

plt.tight_layout()
plt.savefig(od/'power_curve.pdf')
plt.show()

## $C_{p,max}$

In [ ]:
from enn554.wind import cp_max
lams = np.linspace(0.25,25.0,1000)
Cp_max = [cp_max(λ)['Cp_max'] for λ in lams]
Cp_betz = 16.0/27.0

fig,ax = plt.subplots()
ax.plot(lams,Cp_max,label='Cp_max with swirling')
ax.axhline(y=Cp_betz,ls=':',color='red',label='Betz limit')
ax.set_xlabel(r'Tip speed ratio, $\lambda$')
ax.set_ylabel(r'Power coefficient, $C_{p,max}$')
ax.legend()

## Betz optimium blade
An example from lecture

In [ ]:
from enn554.wind import betz_blade
import pandas as pd
result = betz_blade(7.0,7.0,1.0,r=np.arange(0.1,1.1,0.1))
df = pd.DataFrame(result).set_index('r/R')
df.round(3)

In [ ]:
print(df.to_latex())

## Wake effects

Park model

[1] Jensen, N. O. “A Note on Wind Generator Interaction.” In A Note on Wind Generator Interaction, Report Nos. 87-550-0971–9. Risø National Laboratory, 1983.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(12, 5.5))
ax.set_aspect('equal')
ax.axis('off')

# ── Geometry ─────────────────────────────────────────────────────
xL    = 3.0      # x of left disc
xR    = 9.0     # x of right disc
Hd    = 2.1      # half-height of discs
r0    = 0.4     # wake radius at left disc
rR    = r0*3     # wake radius at right disc
slope = (rR - r0) / (xR - xL)
x_virt = xL - r0 / slope   # virtual convergence point

# ── Dashed centreline ─────────────────────────────────────────────
ax.plot([-0.3, xR + 2.4], [0, 0], 'k-', lw=0.8, dashes=(6, 4))

# ── Wake boundary lines ───────────────────────────────────────────
x_far = xR + 2.0
y_far = r0 + slope * (x_far - xL)
ax.plot([x_virt, x_far], [0,  y_far], 'k-', lw=1.0)
ax.plot([x_virt, x_far], [0, -y_far], 'k-', lw=1.0)

# ── lines ────────────────────────────────────────────────────
for xd in [xL, xR]:
    ax.plot([xd, xd], [-Hd, Hd], 'k-', lw=1.0)

# ── Simple arrows (no tail bars, no hatch) ────────────────────────
def draw_arrows(ax, x_tip, y_vals, length):
    for y in y_vals:
        ax.annotate('', xy=(x_tip+length, y), xytext=(x_tip, y),
                    arrowprops=dict(arrowstyle='->', color='k', lw=0.8,
                                   mutation_scale=9))

n_arrows  = 40
free_ratio_L = r0/Hd
free_ratio_R = rR/Hd
y_arrows_free_L  = np.r_[np.linspace(-Hd, -r0, int(n_arrows*(1-free_ratio_L)/2)),
                         np.linspace(r0, Hd, int(n_arrows*(1-free_ratio_L)/2))]
y_arrows_free_R  = np.r_[np.linspace(-Hd, -rR, int(n_arrows*(1-free_ratio_R)/2)),
                         np.linspace(rR, Hd, int(n_arrows*(1-free_ratio_R)/2))]
y_arrows_wake_L  = np.linspace(-r0, r0, int(n_arrows*free_ratio_L))
y_arrows_wake_R  = np.linspace(-rR, rR, int(n_arrows*free_ratio_R))
L_full    = 0.7
L_wakeR    = 0.5
L_wakeL    = 0.35

# Left disc – short in wake, full outside
draw_arrows(ax, xL - 0.02, y_arrows_wake_L, L_wakeL)
draw_arrows(ax, xL - 0.02, y_arrows_free_L, L_full)

ax.plot([xL+L_full-0.05,xL+L_full-0.05],[r0,Hd], 'k-', lw=1.0)
ax.plot([xL+L_full-0.05,xL+L_full-0.05],[-Hd,-r0], 'k-', lw=1.0)
ax.plot([xL+L_wakeL-0.05,xL+L_wakeL-0.05],[-r0,r0], 'k-', lw=1.0)

# x lines
ax.plot([xL,xL],[-Hd-1,-Hd-0.1], 'k-', lw=1.0)
ax.plot([xR,xR],[-Hd-1,-Hd-0.1], 'k-', lw=1.0)

# u lines
ax.plot([xL,xL],[Hd+0.1,Hd+0.5], 'k-', lw=1.0)
ax.plot([xR,xR],[Hd+0.1,Hd+0.5], 'k-', lw=1.0)
ax.plot([xL+L_full-0.05,xL+L_full-0.05],[Hd+0.1,Hd+0.5], 'k-', lw=1.0)
ax.plot([xR+L_full-0.05,xR+L_full-0.05],[Hd+0.1,Hd+0.5], 'k-', lw=1.0)

# Right disc – short in wake, full outside
draw_arrows(ax, xR - 0.02, y_arrows_wake_R, L_wakeR)
draw_arrows(ax, xR - 0.02, y_arrows_free_R, L_full)

ax.plot([xR+L_full-0.05,xR+L_full-0.05],[rR,Hd], 'k-', lw=1.0)
ax.plot([xR+L_full-0.05,xR+L_full-0.05],[-Hd,-rR], 'k-', lw=1.0)
ax.plot([xR+L_wakeR-0.05,xR+L_wakeR-0.05],[-rR,rR], 'k-', lw=1.0)

# ── u label + opposing arrows above each disc ─────────────────────
for xd in [xL, xR]:
    yt = Hd + 0.42
    # arrow pointing right (away from disc)
    ax.annotate('', xy=(xd+0.04, yt), xytext=(xd - 0.54, yt),
                arrowprops=dict(arrowstyle='->', color='k', lw=0.9, mutation_scale=11))
    # arrow pointing left (toward disc)
    ax.annotate('', xy=(xd+L_full-0.07, yt), xytext=(xd +L_full + 0.57, yt),
                arrowprops=dict(arrowstyle='->', color='k', lw=0.9, mutation_scale=11))
    ax.text(xd+0.5*L_full - 0.05, yt-0.2, r'$u_{\infty}$', ha='center', va='bottom', fontsize=14)

# ── v_o and v_x labels ────────────────────────────────────────────
ax.text(xL + L_wakeL + 0.1, 0.02, r'$u_0$', fontsize=13, ha='left', va='bottom')
ax.text(xR + L_wakeR + 0.1, 0.02, r'$u(x)$', fontsize=13, ha='left', va='bottom')

# ── r_0 dimension ─────────────────────────────────────────────────
xd_r0 = xL + 1.0
ax.annotate('', xy=(xd_r0, r0), xytext=(xd_r0, 0.0),
            arrowprops=dict(arrowstyle='<->', color='k', lw=0.9, mutation_scale=9))
ax.text(xd_r0 + 0.1, r0 / 2, r'$r_o$', fontsize=12, va='center')
ax.plot([xL+L_full+0.05,xd_r0+0.5],[r0,r0],'k-',lw=0.5)

# ── r dimension ───────────────────────────────────────────────────
xd_r = xR + L_wakeR + 0.6
ax.annotate('', xy=(xd_r, rR), xytext=(xd_r, 0.0),
            arrowprops=dict(arrowstyle='<->', color='k', lw=0.9, mutation_scale=9))
ax.text(xd_r + 0.12, rR / 2 + 0.08, r'$r(x) = \alpha x + r_o$', fontsize=12, va='center')
ax.plot([xR+L_full+0.05,xd_r +0.5],[rR,rR],'k-',lw=0.5)

# ── x dimension ───────────────────────────────────────────────────
y_x = -Hd - 0.55
ax.annotate('', xy=(xR, y_x), xytext=(xL, y_x),
            arrowprops=dict(arrowstyle='<->', color='k', lw=0.9, mutation_scale=11))
ax.text((xL + xR) / 2, y_x - 0.23, r'$x$', ha='center', fontsize=14)

ax.set_xlim(-0.5, xR + 3.5)
ax.set_ylim(-Hd - 1.1, Hd + 1.1)
plt.tight_layout(pad=0.2)
plt.savefig(od/'wake_clean.png', bbox_inches='tight', dpi=200,
            facecolor='white')
print("Done")

### Area of overlap

In [ ]:
from enn554.math import Circle, circle_overlap_area

c1, c2 = Circle(0,0,2),Circle(np.sqrt(2+1.0),1.0,2)
overlap = circle_overlap_area(c1,c2)
print(f'Area 1: {c1.area():.2f}')
print(f'Area 2: {c2.area():.2f}')
print(f'Overlap Area: {overlap:.2f}')